# NB4 - Final Formula-Based Comparison

Authoritative comparison notebook. It loads B0-B3 final artifacts and refuses to treat missing formula metadata as final evidence.

## Snellius Pipeline Mode

This notebook is now a reporting/orchestration notebook. Heavy training and per-day evaluation are executed by Slurm scripts under `scripts/snellius/`; this notebook reads the resulting artifacts.

Expected artifacts:
- `results/b0_test_results.csv`
- `results/b1_test_results.csv`
- `results/b2_test_results.csv`
- `results/b3_test_results.csv`
- `results/comparison_table.csv`
- `results/overall_summary_table.csv`

Run from MobaXterm/Snellius with:

```bash
cd /home/<user>/thesis
export PROJECT_DIR=/home/<user>/thesis
export DATA_DIR=/scratch-shared/<user>/datasets
export CONDA_ENV=mysimenv
bash scripts/snellius/submit_final_pipeline.sh
```

In [ ]:
import json
import pathlib
import sys

import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = next(
    (p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "procs").exists()),
    pathlib.Path.cwd(),
)
RESULTS_DIR = PROJECT_ROOT / "results"
MODELS_DIR = PROJECT_ROOT / "models"
META_DIR = RESULTS_DIR / "job_metadata"

pd.set_option("display.float_format", "{:.6f}".format)


def require_file(path, label=None, strict=False):
    path = pathlib.Path(path)
    if path.exists():
        print(f"OK      {label or path.name}: {path}")
        return True
    message = f"MISSING {label or path.name}: {path}"
    if strict:
        raise FileNotFoundError(message)
    print(message)
    return False


def read_csv(path, **kwargs):
    path = pathlib.Path(path)
    require_file(path, strict=True)
    return pd.read_csv(path, **kwargs)


def metric_summary(df):
    metrics = [c for c in ["Sharpe", "Sortino", "Max DD", "P&L-to-MAP", "Final PnL", "Mean |q|", "Near Cap Fraction"] if c in df.columns]
    return pd.DataFrame({"Mean": df[metrics].mean(), "Median": df[metrics].median(), "Std": df[metrics].std(ddof=1)})


def formula_metadata_files():
    if not META_DIR.exists():
        return []
    return sorted(META_DIR.glob("*.json"))

print(f"Project root : {PROJECT_ROOT}")
print(f"Results dir  : {RESULTS_DIR}")
print(f"Models dir   : {MODELS_DIR}")
print("Official Snellius execution path: bash scripts/snellius/submit_final_pipeline.sh")

## Artifact Preflight

In [ ]:
require_file(RESULTS_DIR / 'b0_test_results.csv')
require_file(RESULTS_DIR / 'b1_test_results.csv')
require_file(RESULTS_DIR / 'b2_test_results.csv')
require_file(RESULTS_DIR / 'b3_test_results.csv')
require_file(RESULTS_DIR / 'comparison_table.csv')
require_file(RESULTS_DIR / 'overall_summary_table.csv')

## Formula Metadata Check

In [ ]:
meta_files = formula_metadata_files()
formula_meta = []
for path in meta_files:
    try:
        payload = json.loads(path.read_text())
    except Exception:
        continue
    if "formula_as_parameter_ppo" in json.dumps(payload):
        formula_meta.append(path)

if not formula_meta:
    raise FileNotFoundError("No formula-AS job metadata found. Run scripts/snellius/submit_final_pipeline.sh before using this as final evidence.")
print(f"Formula metadata files found: {len(formula_meta)}")
for path in formula_meta[-5:]:
    print(path.name)

## Load B0-B3 Tables

In [ ]:
dfs = {name: read_csv(RESULTS_DIR / f"{name.lower()}_test_results.csv") for name in ["B0", "B1", "B2", "B3"]}
base_dates = dfs["B0"]["Day"].astype(str).tolist()
for name, df in dfs.items():
    dates = df["Day"].astype(str).tolist()
    if dates != base_dates:
        raise ValueError(f"{name} date index does not align with B0")

summary = read_csv(RESULTS_DIR / "overall_summary_table.csv") if (RESULTS_DIR / "overall_summary_table.csv").exists() else pd.DataFrame()
comparison = read_csv(RESULTS_DIR / "comparison_table.csv")
display(comparison)
display(summary)

## Key Comparison Plots

In [ ]:
metrics = ["Sharpe", "Sortino", "Max DD", "Final PnL"]
for metric in metrics:
    table = pd.DataFrame({name: df[metric].values for name, df in dfs.items()}, index=base_dates)
    ax = table.plot(figsize=(13, 4), marker="o", title=metric)
    ax.set_xlabel("Test day")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f"nb4_{metric.replace(' ', '_').replace('&', 'and')}_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()